In [1]:
import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


In [3]:
df = pd.read_csv('../raw/Antioquia.csv', sep= ';', low_memory=False)

In [5]:
df[['ruta', 'zona', 'canal', 'centro_costos','centrocostos','unidadnegocios','troevendedor']] = df[['ruta', 'zona', 'canal', 'centro_costos','centrocostos','unidadnegocios','troevendedor',]].astype(object)
df['fechafactura'] = pd.to_datetime(df['fechafactura'])
df['cliente'] = df['cliente'].astype(int)

In [17]:
def obtener_moda(serie):
    return serie.mode()[0]
df.groupby('cliente')['tro_e_marca'].agg(obtener_moda)


cliente
295436             GP
356663         TRONEX
387708         TRONEX
492108             SL
506032             SL
                ...  
10351286301    TRONEX
10400457777    TRONEX
10413271144    TRONEX
10427700689    TRONEX
10483221957    TRONEX
Name: tro_e_marca, Length: 22694, dtype: object

In [26]:
last_date = df['fechafactura'].max()
df_clientes = df.groupby('cliente').agg(
    frecuencia = ('factura', 'nunique'),
    recencia = ('fechafactura', (lambda x:(last_date - x.max()).days)),
    ticket_promedio = ('montoventapesos', 'mean'),   
    ticket = ('montoventapesos', 'sum'),
    referencias_unicas = ('referencia','nunique'),
    margen_promeido = ('margenpesos', 'mean'),
    margen = ('margenpesos', 'sum'),
    avg_descuento = ('porcentajedescuento','mean')
    
).reset_index(drop= False)

df_clientes['cl_value'] = df_clientes['frecuencia'] * df_clientes['ticket_promedio']

df_lineas = df.groupby('cliente')['lineatat'].agg(obtener_moda)
df_clientes = pd.merge(df_clientes, df_lineas, on= 'cliente', how= 'inner')

df_marcas = df.groupby('cliente')['tro_e_marca'].agg(obtener_moda)
df_clientes = pd.merge(df_clientes, df_marcas, on= 'cliente', how= 'inner')

df_clientes = df_clientes.rename(columns= {'tro_e_marca':'marca_favorita',
                                           'lineatat':'linea_favorita'})
df_clientes.head()

,cliente,frecuencia,recencia,ticket_promedio,ticket,referencias_unicas,margen_promeido,margen,avg_descuento,cl_value,linea_favorita,marca_favorita
0,295436,21,26,"18,500.83","1,239,555.80",24,"10,391.03","696,199.31",0.00,"388,517.49",OTROS,GP
1,356663,11,403,"27,042.66","703,109.06",13,"14,137.01","367,562.19",0.00,"297,469.22",MANGANESO,TRONEX
2,387708,6,310,"22,260.95","267,131.44",4,"11,583.23","138,998.71",0.00,"133,565.72",MANGANESO,TRONEX
3,492108,2,662,"22,092.40","66,277.20",2,"12,850.31","38,550.92",0.00,"44,184.80",ENCENDEDORES,SL
4,506032,5,456,"13,271.72","79,630.32",4,"7,144.44","42,866.62",0.00,"66,358.60",ENCENDEDORES,SL


In [27]:
df_clientes.head()

,cliente,frecuencia,recencia,ticket_promedio,ticket,referencias_unicas,margen_promeido,margen,avg_descuento,cl_value,linea_favorita,marca_favorita
0,295436,21,26,"18,500.83","1,239,555.80",24,"10,391.03","696,199.31",0.00,"388,517.49",OTROS,GP
1,356663,11,403,"27,042.66","703,109.06",13,"14,137.01","367,562.19",0.00,"297,469.22",MANGANESO,TRONEX
2,387708,6,310,"22,260.95","267,131.44",4,"11,583.23","138,998.71",0.00,"133,565.72",MANGANESO,TRONEX
3,492108,2,662,"22,092.40","66,277.20",2,"12,850.31","38,550.92",0.00,"44,184.80",ENCENDEDORES,SL
4,506032,5,456,"13,271.72","79,630.32",4,"7,144.44","42,866.62",0.00,"66,358.60",ENCENDEDORES,SL


In [31]:
df_clientes['linea_favorita'].value_counts()

linea_favorita
MANGANESO       13017
ALKALINAS        2368
OTROS            2088
BOMBILLAS        1935
ENCENDEDORES     1879
TERCEROS         1407
Name: count, dtype: int64